In [ ]:
from langchain_neo4j import Neo4jGraph

NEO4J_URL = "bolt://localhost:7687"
NEO4J_USERNAME = "neo4j"
NEO4J_PASSWORD = "password"  # noqa: S105
TESTS_FILE = "../resources/data/test.ttl"

store = Neo4jGraph(
    url=NEO4J_URL,
    username=NEO4J_USERNAME,
    password=NEO4J_PASSWORD,
)

In [ ]:
import neo4j
from langchain_core.documents import Document

from ontologx.store import GraphDocument, Node, Relationship


def get_subgraph_from_node(node_uri: str, props_to_remove: list[str] | None = None) -> GraphDocument:
    """Get the subgraph of a node in the store.

    The subgraph will contain all the nodes and relationships connected to the given node, even indirectly.
    """
    if props_to_remove is None:
        props_to_remove = []

    props_to_remove = [*props_to_remove, "runName", "embedding"]

    # Ugly but quite efficient. Also filters out the embedding property and the Resource label.
    nodes_subgraphs = store.query(
        """
        MATCH (n {uri: $node_uri})
        CALL apoc.path.subgraphAll(n, {labelFilter: '-Dataset'})
        YIELD nodes, relationships
        RETURN
        [node IN nodes | {
        uri: node.uri,
        type: HEAD([label IN LABELS(node) WHERE label <> 'Resource']),
        properties: apoc.map.removeKeys(PROPERTIES(node), $props_to_remove)
        }] AS nodes,
        [rel IN relationships | {
        source: STARTNODE(rel).uri,
        target: ENDNODE(rel).uri,
        type: TYPE(rel)
        }] AS relationships
        """,
        params={"node_uri": node_uri, "props_to_remove": props_to_remove},
    )

    if not nodes_subgraphs:
        return GraphDocument(nodes=[], relationships=[])

    nodes_subgraph = nodes_subgraphs[0]

    # The neo4j date and time objects are quite problematic, as they are not JSON serializable.
    # This is a workaround to convert them to strings.
    for node in nodes_subgraph["nodes"]:
        for key, value in node["properties"].items():
            if isinstance(value, neo4j.time.DateTime | neo4j.time.Date | neo4j.time.Time):
                node["properties"][key] = value.iso_format()

    def get_node_id(uri: str) -> str:
        """Get the node id from the node uri."""
        final_str = uri.split("/")[-1]

        if "#" in final_str:
            return final_str.split("#")[-1]

        return final_str

    nodes_dict = {
        node["uri"]: Node(id=get_node_id(node["uri"]), type=node["type"], properties=node["properties"])
        for node in nodes_subgraph["nodes"]
    }

    relationships = (
        [
            Relationship(
                source=nodes_dict[relationship["source"]],
                target=nodes_dict[relationship["target"]],
                type=relationship["type"],
            )
            for relationship in nodes_subgraph["relationships"]
        ]
        if "relationships" in nodes_subgraph
        else []  # The node may not have any relationships
    )

    return GraphDocument(
        nodes=list(nodes_dict.values()),
        relationships=relationships,
    )


def tests() -> list[GraphDocument]:
    test_nodes = store.query(
        """
        MATCH (d:Dataset)-[:hasPart]->(e:Event)
        WHERE d.uri STARTS WITH $tests_uri
        RETURN e.eventMessage as message, e.uri as uri
        ORDER BY e.uri
        """,
    )

    tests = []
    for test in test_nodes:
        graph = get_subgraph_from_node(test["uri"])

        source_node = next((node for node in graph.nodes if node.type == "Source"), None)
        context = (
            {
                "sourceName": source_node.properties["sourceName"],
                "sourceDevice": source_node.properties["sourceDevice"],
            }
            if source_node
            else {}
        )
        graph.source = Document(page_content=test["message"], metadata=context)
        tests.append(graph)

    return tests

In [ ]:
# get all generated graphs
y_true = tests()

y_pred = []
for graph in y_true:
    # Get the event node
    event_node = next((node for node in graph.nodes if node.type == "Event"), None)
    if event_node is None:
        msg = "No Event node found in the graph"
        raise ValueError(msg)

    # Get the corresponding pred graph
    pred_event_node_uri = store.query(
        """
        MATCH (d:Dataset)-[:hasPart]->(e:Event)
        WHERE e.uri = $event_uri
        RETURN d.uri as uri
        """,
    )


# get corresponding baseline graphs

# Re-compute metrics